# Can LLMs Find Answers Without Vector Search?

I built two RAG pipelines and tested them head-to-head on 4 questions about the DeepSeek-R1 paper.

**The two approaches:**
- **Vector RAG** — classic chunk-and-embed. Splits the PDF into pieces, converts them to numbers, retrieves the closest matches to the question.
- **PageIndex** — no embeddings. An LLM reads a map of the document and reasons about which sections are relevant.

Both get the same 4 questions. I score every answer manually against the source paper. Let's see which one actually works better and *where*.

> Paper used: [DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning](https://arxiv.org/abs/2501.12948)


## Step 1 — Setup

Load environment variables (API keys from `.env`) and confirm everything is ready.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from rag_shootout import config

missing = config.validate_env()
if missing:
    print(f"Missing keys: {missing}")
    print("Copy .env.example to .env and fill in your API keys.")
else:
    print("API keys loaded.")

print(f"LLM model  : {config.MODEL}")
print(f"Embeddings : {config.EMBEDDING_MODEL}")
print(f"Chunk size : {config.CHUNK_SIZE} chars, overlap={config.CHUNK_OVERLAP}, top_k={config.TOP_K}")


API keys loaded.
LLM model  : qwen/qwen3.5-122b-a10b
Embeddings : all-MiniLM-L6-v2
Chunk size : 800 chars, overlap=150, top_k=5


## Step 2 — Get the Paper

Download the DeepSeek-R1 PDF and extract text from every page. The paper is 20+ pages of dense ML content — a good stress test for both retrieval methods.


In [2]:
from rag_shootout.pdf_utils import download_pdf, extract_pages, chunk_pages

pdf_path = download_pdf()
pages = extract_pages(pdf_path)

print(f"\nFirst 300 characters of page 1:")
print(pages[0]["text"][:300])


[PDF] Using cached file: C:\Users\hp\Downloads\rag-shootout (1)\rag-shootout\sample_paper.pdf (4916 KB)
[PDF] Extracted text from 86 pages (sample_paper.pdf)

First 300 characters of page 1:
DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via
Reinforcement Learning
DeepSeek-AI
research@deepseek.com
Abstract
General reasoning represents a long-standing and formidable challenge in artificial intelli-
gence. Recent breakthroughs, exemplified by large language models (LLMs) (Brown e


## Step 3 — Chop the PDF into Chunks (Vector RAG only)

Vector RAG needs the text broken into small pieces before it can embed them. I'm using 800-character chunks with 150-character overlap so sentences don't get cut off at boundaries.

PageIndex doesn't need this step — it works on the whole document.


In [3]:
import sys
print(sys.executable)

c:\Users\hp\Downloads\rag-shootout (1)\rag-shootout\venv\Scripts\python.exe


In [4]:
chunks = chunk_pages(pages)

print(f"\nExample chunk (chunk #5):")
print("-" * 60)
print(chunks[5]["text"])
print("-" * 60)
print(f"From page {chunks[5]['page']}")


[Chunk] Created 440 chunks (size=800, overlap=150) from 86 pages

Example chunk (chunk #5):
------------------------------------------------------------
provided exemplars, which prevents the exploration of superior, non-human-like reasoning pathways. To tackle these issues, we aim to explore the potential of LLMs for developing reasoning abilities through self-evolution in an RL framework, with minimal reliance on human labeling efforts. Specifically, we build upon DeepSeek-V3-Base (DeepSeek-AI, 2024b) and employ Group Relative Policy Optimization (GRPO) (Shao et al., 2024) as our RL framework. The reward signal is solely based on the correctness of final predictions against ground-truth answers, without imposing constraints on the reasoning process itself. Notably, we bypass the conventional supervised fine-tuning (SFT) phase before RL training. This design choice stems from our hypothesis that human-defined reasoning patterns may limit 
----------------------------------------------

## Step 4 — Build the Vector RAG Pipeline

This embeds all the chunks using `all-MiniLM-L6-v2` (a small but solid sentence embedding model). When a question comes in, it embeds the question the same way and finds the 5 chunks with the highest cosine similarity.

The embedding step takes about 30–60 seconds on first run (downloading the model). After that it's fast.


In [5]:
from rag_shootout.vector_pipeline import VectorRAGPipeline

vector = VectorRAGPipeline()
vector.index(chunks)


c:\Users\hp\Downloads\rag-shootout (1)\rag-shootout\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[VectorRAG] Loading embedder: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10156.89it/s]


[VectorRAG] Embedding 440 chunks ...


Batches: 100%|██████████| 14/14 [00:13<00:00,  1.01it/s]

[VectorRAG] Index ready — shape: (440, 384)


Quick test to make sure retrieval is working:

In [6]:
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path("..") / ".env")

True

In [7]:
from rag_shootout import config

print(config.OPENROUTER_API_KEY[:12] + "...")

sk-or-v1-9e2...


In [8]:
result = vector.answer("What reinforcement learning algorithm did they use?")

print(f"Retrieved from pages: {result['pages_retrieved']}")
print(f"Response time: {result['time_sec']}s")
print()
print(result["answer"])


Retrieved from pages: [1, 2, 14, 63, 65]
Response time: 46.124s




## Step 5 — Set Up PageIndex

This uploads the PDF to PageIndex, which builds a tree structure of the document. That structure is what the LLM reasons over when deciding which sections to retrieve.

**This takes 1–3 minutes on first upload.** After that you can reuse the `doc_id` to skip re-uploading.


In [11]:
from rag_shootout.pageindex_pipeline import PageIndexPipeline

pageindex = PageIndexPipeline()
pageindex.submit(pdf_path)


ImportError: pageindex is not installed. Run: pip install pageindex

Quick test:

In [ ]:
result_pi = pageindex.answer("What reinforcement learning algorithm did they use?")

print(f"Response time: {result_pi['time_sec']}s")
print()
print(result_pi["answer"])


## Step 6 — The 4 Questions

These aren't random questions. Each one is chosen to stress a specific weakness in retrieval systems:

| Type | The challenge |
|---|---|
| Simple factual | Baseline — if both fail here, something's wrong |
| Exact number | Numbers often get split across chunk boundaries |
| Full pipeline | Answer is spread across 5+ sections |
| Buried story | One paragraph, deep in the paper |


In [ ]:
from rag_shootout.questions import get_questions

questions = get_questions()
for q in questions:
    print(f"Q{q.id:2d}  [{q.category}]")
    print(f"    {q.text}")
    print()


## Step 7 — Run the Comparison

Both pipelines answer all 4 questions. This will take a few minutes because of API latency. I print both answers side by side so you can read them against the paper and score them in the next step.


In [ ]:
import time

results = []

for q in questions:
    print("=" * 70)
    print(f"Q{q.id}  [{q.category}]")
    print(q.text)
    print("=" * 70)

    v = vector.answer(q.text)
    print(f"\nVECTOR RAG  ({v['time_sec']}s | pages {v['pages_retrieved']})")
    print(v["answer"][:600])

    p = pageindex.answer(q.text)
    print(f"\nPAGEINDEX  ({p['time_sec']}s)")
    print(p["answer"][:600])
    print()

    results.append({
        "question_id": q.id,
        "category": q.category,
        "question": q.text,
        "vector_answer": v["answer"],
        "vector_time": v["time_sec"],
        "vector_pages": v["pages_retrieved"],
        "pageindex_answer": p["answer"],
        "pageindex_time": p["time_sec"],
    })

    time.sleep(2)

print(f"Done — {len(results)} questions answered by both pipelines.")


## Step 8 — Score Each Answer

Open the paper in another tab and score every answer 1–5 on three things:

| Dimension | 1 means... | 5 means... |
|---|---|---|
| **Accuracy** | The answer is factually wrong | Everything is correct |
| **Completeness** | It missed the point | It covered everything asked |
| **Faithfulness** | It made things up | Every claim is in the paper |

Fill in the scores below, then re-run the cells underneath to see the results.


In [ ]:
from rag_shootout.scoring import make_empty_scorecard, DimensionScore, scores_to_dataframe, print_summary

scorecard = make_empty_scorecard(n_questions=4)

# ── Fill in your scores here ──────────────────────────────────────────────
scorecard[0].vector    = DimensionScore(accuracy=5, completeness=4, faithfulness=5)
scorecard[0].pageindex = DimensionScore(accuracy=1, completeness=1, faithfulness=1)

scorecard[1].vector    = DimensionScore(accuracy=1, completeness=1, faithfulness=4)
scorecard[1].pageindex = DimensionScore(accuracy=1, completeness=1, faithfulness=1)

scorecard[2].vector    = DimensionScore(accuracy=2, completeness=1, faithfulness=4)
scorecard[2].pageindex = DimensionScore(accuracy=1, completeness=1, faithfulness=1)

scorecard[3].vector    = DimensionScore(accuracy=5, completeness=5, faithfulness=5)
scorecard[3].pageindex = DimensionScore(accuracy=1, completeness=1, faithfulness=1)
# ─────────────────────────────────────────────────────────────────────────

scored = sum(1 for s in scorecard if s.vector.is_complete() and s.pageindex.is_complete())
print(f"Scored so far: {scored} / {len(scorecard)} questions")
print("Scores filled. Run the next cells to see the summary and table.")


In [ ]:
df = scores_to_dataframe(scorecard, results)
print_summary(scorecard)


In [ ]:
df[["category", "winner", "vector_total", "pageindex_total", "vector_time", "pageindex_time"]]


## Step 9 — Charts

Four views of the same data. The most interesting one is usually the last — wins broken down by question type, which shows *where* each approach is actually better.


In [ ]:
import matplotlib.pyplot as plt
from rag_shootout.visualization import (
    plot_scores_per_question,
    plot_win_counts,
    plot_dimension_averages,
    plot_latency,
    plot_wins_by_category,
)

for fn in [plot_scores_per_question, plot_win_counts, plot_dimension_averages, plot_latency, plot_wins_by_category]:
    try:
        fig = fn(df)
        plt.show()
    except (ValueError, KeyError) as e:
        print(f"Skipped {fn.__name__} — score more questions first. ({e})")


## Step 10 — Save Results

In [ ]:
from rag_shootout.config import RESULTS_DIR

out = RESULTS_DIR / "benchmark_results.csv"
df.to_csv(out, index=False)
print(f"Saved to {out}")


## My Takeaways

*(Fill this in after running and scoring)*

**Overall winner:**

**Vector RAG was better when:**

**PageIndex was better when:**

**The result that surprised me most:**

**Would I use the faster approach in production, or is the accuracy gap too big?**

**When would I actually reach for each one in a real project?**

---

*If you want to try this on a different paper, change `PDF_URL` in `src/rag_shootout/config.py` and swap in your own questions in `src/rag_shootout/questions.py`.*
